In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from src.dataset import GWHDDensityDataset, IMAGENET_MEAN, IMAGENET_STD
from src.utils import set_seed

DATA = ROOT / "data" / "raw" / "zenodo" / "gwhd_2021.zip"  # oryginalne wydanie z Zenodo


AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [ ]:
set_seed(0)
train = GWHDDensityDataset(DATA, "train", sigma=10, crop_size=256)
val = GWHDDensityDataset(DATA, "validation", sigma=10)  # pełne obrazy

s = train[0]
print("image:  ", tuple(s["image"].shape), s["image"].dtype)
print("density:", tuple(s["density"].shape))
print("count:  ", s["count"].item(), "| suma mapy:", round(s["density"].sum().item(), 4))


In [ ]:
def show(sample):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = (sample["image"] * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()  # (H, W, 3)
    dens = sample["density"][0].numpy()

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(img)
    ax[0].set_title(f"{sample['domain']} | kłosów: {sample['count'].item():.0f}")
    # extent rozciąga małą mapę (32x32) na rozmiar obrazu (256x256)
    ax[1].imshow(dens, cmap="jet", extent=(0, img.shape[1], img.shape[0], 0))
    ax[1].set_title(f"mapa {dens.shape}, suma = {dens.sum():.2f}")
    for a in ax:
        a.axis("off")
    plt.show()

show(train[0])   # losowy wycinek z train
show(val[0])     # cały obraz z walidacji


In [ ]:
samples = [train[i] for i in range(200)]  # każdą próbkę losujemy raz (wycinek jest losowy)
diffs = [abs(s["density"].sum().item() - s["count"].item()) for s in samples]
print("największa różnica suma vs count:", max(diffs))

empty = next(i for i, p in enumerate(val.points) if len(p) == 0)
print("obraz bez kłosów nr", empty, "-> count:", val[empty]["count"].item(), "suma:", val[empty]["density"].sum().item())


In [ ]:
show(GWHDDensityDataset(DATA, "validation", sigma=4)[0])    # węższe plamy
show(GWHDDensityDataset(DATA, "validation", sigma=25)[0])   # szersze plamy
GWHDDensityDataset(DATA, "validation", sigma=10, crop_size=100)  # celowy błąd


# Testy zadania 3: model CSRNet i trening testowy

Te komórki wymagają tylko **komórki 1** (importy i `ROOT`). Po zmianach w `src/` zrób **Restart** kernela.

Trening uruchamia się w terminalu (nie w notebooku):
```powershell
.\.venv\Scripts\Activate.ps1
python train.py --config configs/smoke.yaml
```
Wyniki trafiają do `outputs/smoke/` (folder jest w `.gitignore`).

### Test A: kształty wyjścia modelu

CSRNet zmniejsza obraz 8 razy. **Oczekiwane:** 256 → 32, 1024 → 128 oraz ok. 16,26 mln parametrów.
Pierwsze uruchomienie może trwać dłużej (wagi VGG16 są już pobrane do `~/.cache/torch`).

In [ ]:
from src.model import CSRNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CSRNet(pretrained=True).to(device).eval()
print("urządzenie:", device, "| parametry:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2), "mln")

with torch.no_grad():
    for size in (256, 1024):
        out = model(torch.randn(1, 3, size, size, device=device))
        print(f"wejście (1, 3, {size}, {size}) -> wyjście {tuple(out.shape)}")
        assert out.shape[-1] == size // 8

### Test B: historia treningu

`history.csv` zawiera wiersz na każdą epokę. **Oczekiwane:**
- `train_loss` wyraźnie spada,
- `train_mae` (błąd liczby kłosów na **obrazach treningowych**) spada najszybciej - model „uczy się na pamięć" 32 obrazów,
- `val_mae` (obrazy walidacyjne z innych krajów) może spadać wolniej albo wcale - przy 32 obrazach to normalne.

Trening testowy sprawdza **działanie kodu**, a nie jakość modelu - tych liczb nie raportujemy w pracy.

In [ ]:
import pandas as pd

RUN = ROOT / "outputs" / "smoke"
hist = pd.read_csv(RUN / "history.csv")
display(hist.dropna(subset=["val_mae"]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["epoch"], hist["train_loss"])
ax[0].set(title="strata treningowa (MSE)", xlabel="epoka", yscale="log")
ev = hist.dropna(subset=["val_mae"])
ax[1].plot(ev["epoch"], ev["train_mae"], "o-", label="train (32 obrazy)")
ax[1].plot(ev["epoch"], ev["val_mae"], "o-", label="validation (16 obrazów)")
ax[1].set(title="MAE liczby kłosów na obraz", xlabel="epoka")
ax[1].legend()
plt.show()

assert hist["train_loss"].iloc[-5:].mean() < hist["train_loss"].iloc[:5].mean(), "strata nie spada!"

### Test C: predykcje per obraz

`predictions_val_best.csv` - dla każdego obrazu walidacyjnego: prawdziwa i przewidziana liczba kłosów
(z najlepszej epoki). Na wykresie punkty powinny układać się **wzdłuż przekątnej**; im bliżej, tym lepiej.

In [ ]:
pred = pd.read_csv(RUN / "predictions_val_best.csv")
display(pred.round(2))

lim = max(pred["true_count"].max(), pred["pred_count"].max()) * 1.1
plt.figure(figsize=(5, 5))
plt.scatter(pred["true_count"], pred["pred_count"])
plt.plot([0, lim], [0, lim], "k--", label="idealnie")
plt.xlabel("prawdziwa liczba kłosów")
plt.ylabel("przewidziana liczba kłosów")
plt.legend()
plt.show()

print("MAE z pliku:", round(pred["abs_error"].mean(), 3))

### Test D: checkpoint i predykcja na jednym obrazie

Wczytujemy `best.pt` i sprawdzamy, co zawiera. Potem model przewiduje mapę dla jednego obrazu walidacyjnego.

**Skalowanie:** w treningu cel to `mapa × density_scale` (100), więc przewidywana liczba = `suma wyjścia / 100`.

**Oczekiwane:** przewidziana mapa ma plamy w podobnych miejscach co prawdziwa (po krótkim treningu - rozmyte).

In [ ]:
from src.utils import load_checkpoint
from src.dataset import GWHDDensityDataset

ckpt = load_checkpoint(RUN / "best.pt", map_location=device)
print("klucze checkpointu:", list(ckpt))
print("epoka:", ckpt["epoch"], "| val MAE:", round(ckpt["best_val_mae"], 3))

cfg = ckpt["config"]
model = CSRNet(pretrained=False).to(device).eval()
model.load_state_dict(ckpt["model"])

ds = GWHDDensityDataset(DATA, "validation", sigma=cfg["sigma"], indices=[int(pred["index"].iloc[0])])
s = ds[0]
with torch.no_grad():
    out = model(s["image"].unsqueeze(0).to(device))[0, 0].cpu().numpy() / cfg["density_scale"]

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
img = (s["image"] * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(img); ax[0].set_title(f"obraz val nr {s['index']} ({s['domain']})")
ax[1].imshow(s["density"][0], cmap="jet"); ax[1].set_title(f"prawdziwa mapa: {s['count'].item():.0f} kłosów")
ax[2].imshow(out, cmap="jet"); ax[2].set_title(f"przewidziana mapa: {out.sum():.1f} kłosów")
for a in ax:
    a.axis("off")
plt.show()